# Phase 5 — SHAP Explainability
**UCI Hydraulic Systems Dataset — Predictive Maintenance**

Answers the key question regulators and plant operators ask: **why did the model flag this component?**

Steps covered:
1. Load tuned XGBoost model and test set from Phase 4
2. Compute SHAP values for all 4 targets
3. Global — beeswarm summary plot (top features overall)
4. Global — mean |SHAP| bar chart per sensor group
5. Global — feature importance heatmap across all targets
6. Local — waterfall plot for a single cycle
7. Local — healthy vs faulty cycle comparison
8. Feature group contribution analysis
9. Save SHAP values for dashboard

## 5.0 — Imports & configuration

In [ ]:
from pathlib import Path
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import shap

shap.initjs()

BASE_DIR      = Path.cwd().parent
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
MODELS_DIR    = BASE_DIR / 'models'
SHAP_DIR      = BASE_DIR / 'data' / 'shap'
SHAP_DIR.mkdir(exist_ok=True)

TARGETS = ['cooler', 'valve', 'pump', 'accumulator']

LABEL_NAMES = {
    'cooler':      {0: 'near failure',   1: 'reduced eff.',  2: 'full eff.'},
    'valve':       {0: 'near failure',   1: 'severe lag',    2: 'small lag',     3: 'optimal'},
    'pump':        {0: 'no leakage',     1: 'weak leakage',  2: 'severe leakage'},
    'accumulator': {0: 'near failure',   1: 'severely red.', 2: 'slightly red.', 3: 'optimal'},
}

TARGET_PALETTE = {
    'cooler':      '#7F77DD',
    'valve':       '#1D9E75',
    'pump':        '#EF9F27',
    'accumulator': '#D85A30',
}

SENSOR_GROUPS = {
    'pressure':    ['PS1','PS2','PS3','PS4','PS5','PS6'],
    'motor_power': ['EPS1'],
    'flow':        ['FS1','FS2'],
    'temperature': ['TS1','TS2','TS3','TS4'],
    'vibration':   ['VS1'],
    'efficiency':  ['CE','CP','SE'],
}

print('Configuration loaded ✓')

## 5.1 — Load model and test data

In [ ]:
with open(MODELS_DIR / 'xgb_tuned.pkl', 'rb') as f:
    model = pickle.load(f)

X_test = pd.read_parquet(MODELS_DIR / 'X_test.parquet', engine='fastparquet')
y_test = pd.read_csv(MODELS_DIR / 'y_test.csv')
y_pred = pd.read_csv(MODELS_DIR / 'y_pred_test.csv')

with open(PROCESSED_DIR / 'feature_groups.json') as f:
    feature_groups = json.load(f)

feature_columns = X_test.columns.tolist()
N_TEST          = X_test.shape[0]

print(f'Model:    {type(model.estimators_[0]).__name__} wrapped in MultiOutputClassifier')
print(f'X_test:   {X_test.shape}')
print(f'y_test:   {y_test.shape}')
print(f'Features: {len(feature_columns)}')
print()
print('Feature groups:')
for grp, cols in feature_groups.items():
    print(f'  {grp:<25} {len(cols)} features')

## 5.2 — Compute SHAP values

Modern SHAP (0.46+) returns a **3D array `(n_samples, n_features, n_classes)`**.
Older SHAP returned a **list** of `(n_samples, n_features)` arrays.

`normalise_shap()` converts both formats to a consistent `(n_classes, n_samples, n_features)` array
so all downstream cells work identically regardless of SHAP version.

In [ ]:
def normalise_shap(sv):
    """
    Normalise any SHAP output to shape (n_classes, n_samples, n_features).

    Handles:
      list of (n_samples, n_features)            older multi-class
      ndarray (n_samples, n_features, n_classes)  newer multi-class
      ndarray (n_samples, n_features)             binary / single output
    """
    if isinstance(sv, list):
        return np.stack(sv, axis=0)          # (n_classes, n_samples, n_features)
    elif isinstance(sv, np.ndarray) and sv.ndim == 3:
        return sv.transpose(2, 0, 1)         # (n_samples, n_features, n_classes) -> (n_classes, n_samples, n_features)
    else:
        return sv[np.newaxis, :, :]          # (n_samples, n_features) -> (1, n_samples, n_features)


def get_sv_for_class(target, class_idx):
    """
    Return (n_samples, n_features) SHAP slice for a specific class.
    Clamps class_idx to valid range automatically.
    """
    sv        = shap_values_3d[target]              # (n_classes, n_samples, n_features)
    cls_idx   = min(int(class_idx), sv.shape[0]-1)
    return sv[cls_idx]                              # (n_samples, n_features)


print('Computing SHAP values for all 4 targets...')
print('(TreeExplainer — expect ~10-30 seconds total)\n')

shap_values_3d = {}   # target -> (n_classes, n_samples, n_features)
explainers     = {}   # target -> TreeExplainer
shap_mean_abs  = {}   # target -> pd.Series of mean |SHAP| per feature

for i, target in enumerate(TARGETS):
    estimator = model.estimators_[i]
    explainer = shap.TreeExplainer(estimator)
    sv_raw    = explainer.shap_values(X_test)
    sv        = normalise_shap(sv_raw)    # always (n_classes, n_samples, n_features)

    explainers[target]     = explainer
    shap_values_3d[target] = sv

    # Mean |SHAP| across all classes and all samples -> (n_features,)
    mean_abs = np.abs(sv).mean(axis=(0, 1))
    assert mean_abs.shape == (len(feature_columns),), \
        f'{target}: mean_abs shape {mean_abs.shape} != ({len(feature_columns)},)'

    shap_mean_abs[target] = pd.Series(mean_abs, index=feature_columns)
    print(f'  {target:<14} ✓  {sv.shape[0]} classes  |  sv shape: {sv.shape}')

print('\nSHAP computation complete ✓')

## 5.3 — Global: Beeswarm summary plots (top 15 features per target)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.patch.set_facecolor('#0d1117')
axes = axes.flatten()

rng = np.random.default_rng(42)

for ax, target in zip(axes, TARGETS):
    sv_plot = get_sv_for_class(target, 0)      # (n_samples, n_features) — near-failure class

    mean_abs    = np.abs(sv_plot).mean(axis=0)
    top15_idx   = np.argsort(mean_abs)[-15:].tolist()
    top15_names = [feature_columns[i] for i in top15_idx]
    sv_top15    = sv_plot[:, top15_idx]
    X_top15     = X_test.values[:, top15_idx].astype(float)

    ax.set_facecolor('#161b22')
    y_pos = np.arange(15)

    for j in range(15):
        shap_col  = sv_top15[:, j]
        feat_col  = X_top15[:, j]
        feat_min  = feat_col.min()
        feat_max  = feat_col.max()
        feat_norm = (feat_col - feat_min) / (feat_max - feat_min + 1e-10)
        colors    = plt.cm.RdBu_r(feat_norm)
        jitter    = rng.uniform(-0.3, 0.3, size=len(shap_col))
        ax.scatter(shap_col, y_pos[j] + jitter,
                   c=colors, s=6, alpha=0.6, linewidths=0)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(top15_names, fontsize=7.5, color='#ccc')
    ax.axvline(0, color='#555', linewidth=0.8, linestyle='--')
    ax.set_xlabel('SHAP value (impact on near-failure prediction)', color='#888', fontsize=8)
    ax.set_title(f'{target.capitalize()} — top 15 features (near-failure class)',
                 color='white', fontsize=10, pad=6)
    ax.tick_params(colors='#888', labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#30363d')

    sm = plt.cm.ScalarMappable(cmap='RdBu_r', norm=plt.Normalize(0, 1))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label('Feature value\n(blue=low, red=high)', color='#888', fontsize=7)
    cbar.ax.yaxis.set_tick_params(color='#888', labelsize=7)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#888')

fig.suptitle('SHAP beeswarm — top 15 features per target (near-failure class)',
             color='white', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'plot_shap_beeswarm.png',
            dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 5.4 — Global: Mean |SHAP| bar chart — top 20 features per target

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.patch.set_facecolor('#0d1117')
axes = axes.flatten()

for ax, target in zip(axes, TARGETS):
    color = TARGET_PALETTE[target]
    top20 = shap_mean_abs[target].nlargest(20)

    ax.set_facecolor('#161b22')
    bars = ax.barh(range(20), top20.values[::-1],
                   color=color, alpha=0.85,
                   edgecolor='#1a1a2e', linewidth=0.4)
    ax.set_yticks(range(20))
    ax.set_yticklabels(top20.index[::-1], fontsize=8, color='#ccc')
    ax.set_xlabel('Mean |SHAP value|', color='#888', fontsize=9)
    ax.set_title(f'{target.capitalize()} — top 20 features by importance',
                 color='white', fontsize=10, pad=6)
    ax.tick_params(colors='#888', labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#30363d')

    max_val = top20.values.max()
    for bar, val in zip(bars, top20.values[::-1]):
        ax.text(val + max_val * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', color='#aaa', fontsize=7)

fig.suptitle('Global feature importance — mean |SHAP| across test set',
             color='white', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'plot_shap_importance.png',
            dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 5.5 — Global: Feature group contribution analysis

In [ ]:
def get_engineering_group(feat_name):
    if feat_name.startswith('cross_'):
        return 'Group C: Cross-sensor'
    elif any(s in feat_name for s in ['spec_energy', 'dom_freq', 'spec_centroid']):
        return 'Group B: Frequency domain'
    elif feat_name.endswith('_crest'):
        return 'Group A: Crest factor'
    else:
        return 'Group A: Time-domain stats'


def get_sensor_group(feat_name):
    for group, sensors in SENSOR_GROUPS.items():
        for s in sensors:
            if feat_name.startswith(s + '_') or feat_name.startswith(s.lower() + '_'):
                return group
    return 'cross_sensor'


group_contrib_all  = {t: {} for t in TARGETS}
sensor_contrib_all = {t: {} for t in TARGETS}

for target in TARGETS:
    for feat, val in shap_mean_abs[target].items():
        eg = get_engineering_group(feat)
        group_contrib_all[target][eg]  = group_contrib_all[target].get(eg, 0) + val
        sg = get_sensor_group(feat)
        sensor_contrib_all[target][sg] = sensor_contrib_all[target].get(sg, 0) + val

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0d1117')

x      = np.arange(len(TARGETS))
x_lbls = [t.capitalize() for t in TARGETS]

# Engineering group breakdown
ax1 = axes[0]
ax1.set_facecolor('#161b22')
eng_groups = ['Group A: Time-domain stats', 'Group A: Crest factor',
              'Group B: Frequency domain',  'Group C: Cross-sensor']
eng_colors = ['#7F77DD', '#5B54AA', '#1D9E75', '#EF9F27']
width      = 0.18
offsets    = np.linspace(-0.27, 0.27, len(eng_groups))

for grp, color, offset in zip(eng_groups, eng_colors, offsets):
    vals = [group_contrib_all[t].get(grp, 0) for t in TARGETS]
    ax1.bar(x + offset, vals, width,
            label=grp.replace('Group ', 'Grp '),
            color=color, alpha=0.85, edgecolor='#1a1a2e', linewidth=0.4)

ax1.set_xticks(x)
ax1.set_xticklabels(x_lbls, color='#888', fontsize=10)
ax1.set_ylabel('Summed mean |SHAP|', color='#888', fontsize=9)
ax1.set_title('Contribution by feature engineering group', color='white', fontsize=10, pad=8)
ax1.tick_params(colors='#888')
for spine in ax1.spines.values():
    spine.set_edgecolor('#30363d')
ax1.legend(fontsize=8, labelcolor='white', facecolor='#161b22',
           edgecolor='#30363d', loc='upper right')

# Sensor type breakdown
ax2 = axes[1]
ax2.set_facecolor('#161b22')
sensor_types  = list(SENSOR_GROUPS.keys()) + ['cross_sensor']
sensor_colors = ['#7F77DD','#1D9E75','#EF9F27','#D85A30','#D4537E','#378ADD','#BA7517']
width2        = 0.10
offsets2      = np.linspace(-0.30, 0.30, len(sensor_types))

for stype, color, offset in zip(sensor_types, sensor_colors, offsets2):
    vals = [sensor_contrib_all[t].get(stype, 0) for t in TARGETS]
    ax2.bar(x + offset, vals, width2,
            label=stype, color=color, alpha=0.85,
            edgecolor='#1a1a2e', linewidth=0.4)

ax2.set_xticks(x)
ax2.set_xticklabels(x_lbls, color='#888', fontsize=10)
ax2.set_ylabel('Summed mean |SHAP|', color='#888', fontsize=9)
ax2.set_title('Contribution by sensor type', color='white', fontsize=10, pad=8)
ax2.tick_params(colors='#888')
for spine in ax2.spines.values():
    spine.set_edgecolor('#30363d')
ax2.legend(fontsize=8, labelcolor='white', facecolor='#161b22',
           edgecolor='#30363d', loc='upper right')

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'plot_shap_group_contrib.png',
            dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print('Engineering group totals:\n')
for target in TARGETS:
    total = sum(group_contrib_all[target].values())
    print(f'  {target.capitalize()}:')
    for grp, val in sorted(group_contrib_all[target].items(),
                           key=lambda kv: kv[1], reverse=True):
        pct = 100 * val / total
        print(f'    {grp:<35} {val:.4f}  ({pct:.1f}%)')
    print()

## 5.6 — Global: Cross-target feature importance heatmap

In [ ]:
importance_df = pd.DataFrame(shap_mean_abs).fillna(0)
importance_df['max'] = importance_df.max(axis=1)
top25      = importance_df.nlargest(25, 'max').drop(columns='max')
top25_norm = top25 / (top25.max() + 1e-10)

fig, ax = plt.subplots(figsize=(10, 10))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')

im = ax.imshow(top25_norm.values, cmap='YlOrRd', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(4))
ax.set_xticklabels([t.capitalize() for t in TARGETS], color='white', fontsize=11)
ax.set_yticks(range(25))
ax.set_yticklabels(top25.index, color='#ccc', fontsize=8)

for i in range(25):
    for j in range(4):
        val = top25.values[i, j]
        ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=7,
                color='black' if top25_norm.values[i, j] > 0.5 else '#aaa')

cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('Normalised importance (per target)', color='#888', fontsize=9)
cbar.ax.yaxis.set_tick_params(color='#888')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#888')
ax.set_title('Top 25 features — mean |SHAP| across all targets',
             color='white', fontsize=12, pad=10)
for spine in ax.spines.values():
    spine.set_edgecolor('#30363d')

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'plot_shap_cross_target_heatmap.png',
            dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 5.7 — Local: Waterfall plot — single cycle explanation

In [ ]:
def plot_waterfall(target, cycle_idx, n_features=12):
    pred_class = int(y_pred[target].iloc[cycle_idx])
    true_class = int(y_test[target].iloc[cycle_idx])

    sv_class = get_sv_for_class(target, pred_class)   # (n_samples, n_features)
    sv_cycle = sv_class[cycle_idx]                    # (n_features,)

    top_idx   = np.argsort(np.abs(sv_cycle))[-n_features:].tolist()
    top_shap  = sv_cycle[top_idx]
    top_names = [feature_columns[i] for i in top_idx]
    top_vals  = X_test.values[cycle_idx, top_idx]

    sort_order = np.argsort(top_shap).tolist()
    top_shap   = top_shap[sort_order]
    top_names  = [top_names[i] for i in sort_order]
    top_vals   = top_vals[sort_order]

    fig, ax = plt.subplots(figsize=(11, 7))
    fig.patch.set_facecolor('#0d1117')
    ax.set_facecolor('#161b22')

    colors = ['#D85A30' if v > 0 else '#7F77DD' for v in top_shap]
    bars   = ax.barh(range(n_features), top_shap, color=colors,
                     alpha=0.85, edgecolor='#1a1a2e', linewidth=0.4)
    ax.axvline(0, color='#555', linewidth=0.8)

    labels = [f'{n}  (={v:.3f})' for n, v in zip(top_names, top_vals)]
    ax.set_yticks(range(n_features))
    ax.set_yticklabels(labels, fontsize=8, color='#ccc')

    x_scale = max(abs(top_shap).max() * 0.01, 1e-6)
    for bar, shap_v in zip(bars, top_shap):
        x_pos = shap_v + (x_scale if shap_v >= 0 else -x_scale)
        ha    = 'left' if shap_v >= 0 else 'right'
        ax.text(x_pos, bar.get_y() + bar.get_height() / 2,
                f'{shap_v:+.4f}', va='center', ha=ha,
                color='white', fontsize=7.5)

    correct  = '✓ Correct' if pred_class == true_class else '✗ Wrong'
    pred_lbl = LABEL_NAMES[target][pred_class]
    true_lbl = LABEL_NAMES[target][true_class]

    ax.set_xlabel('SHAP value (contribution to predicted class probability)',
                  color='#888', fontsize=9)
    ax.set_title(
        f'Waterfall — {target.capitalize()} | Cycle {cycle_idx}\n'
        f'Predicted: "{pred_lbl}"  |  Actual: "{true_lbl}"  |  {correct}\n'
        f'Red = pushes toward this prediction  |  Blue = pushes away',
        color='white', fontsize=10, pad=8
    )
    ax.tick_params(colors='#888', labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#30363d')
    ax.legend(
        handles=[
            mpatches.Patch(color='#D85A30', alpha=0.85, label='Increases fault probability'),
            mpatches.Patch(color='#7F77DD', alpha=0.85, label='Decreases fault probability'),
        ],
        fontsize=8, labelcolor='white', facecolor='#161b22', edgecolor='#30363d'
    )
    plt.tight_layout()
    fname = PROCESSED_DIR / f'plot_waterfall_{target}_cycle{cycle_idx}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    print(f'Saved: {fname.name}')


# Find best example cycles (correctly predicted near-failure)
print('Finding correctly-predicted near-failure cycles per target:\n')
example_cycles = {}
for target in TARGETS:
    worst_class = 0
    mask        = (y_test[target] == worst_class) & (y_pred[target] == worst_class)
    candidates  = np.where(mask.values)[0]
    if len(candidates) > 0:
        example_cycles[target] = int(candidates[0])
        lbl = LABEL_NAMES[target][worst_class]
        print(f'  {target:<14} cycle {candidates[0]:>3}  (predicted & true: "{lbl}") ✓')
    else:
        mask2 = (y_test[target] == y_pred[target])
        example_cycles[target] = int(np.where(mask2.values)[0][0])
        print(f'  {target:<14} cycle {example_cycles[target]:>3}  (no worst-class correct — using first correct)')

In [ ]:
for target in TARGETS:
    print(f'\n{"─"*50}  {target.upper()}')
    plot_waterfall(target, example_cycles[target])

## 5.8 — Local: Healthy vs faulty cycle comparison

In [ ]:
def plot_healthy_vs_faulty(target, n_features=10):
    worst_class  = 0
    best_class   = max(LABEL_NAMES[target].keys())

    sv_cls0 = get_sv_for_class(target, worst_class)   # (n_samples, n_features)

    fault_mask   = (y_test[target] == worst_class) & (y_pred[target] == worst_class)
    healthy_mask = (y_test[target] == best_class)  & (y_pred[target] == best_class)

    if not fault_mask.any() or not healthy_mask.any():
        print(f'  Skipping {target} — insufficient examples for comparison')
        return

    fault_idx   = int(np.where(fault_mask.values)[0][0])
    healthy_idx = int(np.where(healthy_mask.values)[0][0])

    sv_fault   = sv_cls0[fault_idx]    # (n_features,)
    sv_healthy = sv_cls0[healthy_idx]  # (n_features,)

    combined  = np.maximum(np.abs(sv_fault), np.abs(sv_healthy))
    top_idx   = np.argsort(combined)[-n_features:][::-1].tolist()
    top_names = [feature_columns[i] for i in top_idx]

    y_pos  = np.arange(n_features)
    height = 0.35

    fig, ax = plt.subplots(figsize=(12, 6))
    fig.patch.set_facecolor('#0d1117')
    ax.set_facecolor('#161b22')

    ax.barh(y_pos + height / 2, sv_fault[top_idx],   height,
            label='Faulty (near failure)',
            color='#D85A30', alpha=0.85, edgecolor='#1a1a2e', linewidth=0.4)
    ax.barh(y_pos - height / 2, sv_healthy[top_idx], height,
            label='Healthy (optimal)',
            color='#1D9E75', alpha=0.85, edgecolor='#1a1a2e', linewidth=0.4)

    ax.axvline(0, color='#555', linewidth=0.8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(top_names, fontsize=8, color='#ccc')
    ax.set_xlabel('SHAP value (near-failure class)', color='#888', fontsize=9)
    ax.set_title(
        f'{target.capitalize()} — healthy vs faulty SHAP comparison\n'
        f'Positive = evidence FOR near-failure prediction',
        color='white', fontsize=10, pad=8
    )
    ax.tick_params(colors='#888', labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#30363d')
    ax.legend(fontsize=9, labelcolor='white',
              facecolor='#161b22', edgecolor='#30363d')

    plt.tight_layout()
    fname = PROCESSED_DIR / f'plot_shap_comparison_{target}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show()


for target in TARGETS:
    print(f'\n{target.upper()}')
    plot_healthy_vs_faulty(target)

## 5.9 — Print top features per target (for presentation slide)

In [ ]:
print('Top 10 most important features per target\n')
print('(Use these on your presentation slides)\n')

unit_map = {
    'PS': 'bar', 'EPS': 'W',   'FS': 'l/min',
    'TS': 'degC', 'VS': 'mm/s', 'CE': '%', 'CP': 'kW', 'SE': '%'
}

for target in TARGETS:
    top10 = shap_mean_abs[target].nlargest(10)
    print(f'  {target.upper()}')
    for rank, (feat, val) in enumerate(top10.items(), 1):
        sensor = feat.split('_')[0] if not feat.startswith('cross') else 'cross'
        unit   = next((m for s, m in unit_map.items() if sensor.startswith(s)), '')
        print(f'    {rank:>2}. {feat:<35} importance={val:.4f}  {unit}')
    print()

## 5.10 — Save SHAP values for dashboard

In [ ]:
# Global importance table
importance_df = pd.DataFrame(shap_mean_abs)
importance_df.to_csv(SHAP_DIR / 'shap_mean_abs_importance.csv')

# SHAP arrays — saved as (n_classes, n_samples, n_features)
for target in TARGETS:
    arr = shap_values_3d[target]
    np.save(SHAP_DIR / f'shap_{target}.npy', arr)

# Expected values (baselines for waterfall plots)
expected_vals = {}
for target in TARGETS:
    ev = explainers[target].expected_value
    expected_vals[target] = ev.tolist() if isinstance(ev, np.ndarray) else float(ev)
with open(SHAP_DIR / 'expected_values.json', 'w') as f:
    json.dump(expected_vals, f, indent=2)

# Top-10 summary
top10_summary = {
    target: shap_mean_abs[target].nlargest(10).to_dict()
    for target in TARGETS
}
with open(SHAP_DIR / 'top10_features.json', 'w') as f:
    json.dump(top10_summary, f, indent=2)

print('Saved to data/shap/:\n')
print('  shap_mean_abs_importance.csv')
for t in TARGETS:
    print(f'  shap_{t}.npy  {shap_values_3d[t].shape}')
print('  expected_values.json')
print('  top10_features.json')

## 5.11 — Summary

**Key findings to present:**

| Target | Top driver | Physical interpretation |
|---|---|---|
| Cooler | TS1/TS2 mean temperature | Degraded cooler → higher operating temp |
| Valve | PS1/PS2 slope or argmax | Lagging valve → pressure peaks later in cycle |
| Pump | FS1 std / flow imbalance | Leaking pump → erratic flow variation |
| Accumulator | PS2 mean / pressure drop | Low accumulator pressure → abnormal PS2 levels |

*(Verify against your actual top feature outputs in cell 5.9)*

**Next → `dashboard/app.py`** (Streamlit)

Build the interactive dashboard with:
- 4-component health traffic light display
- Per-cycle sensor signal viewer
- SHAP waterfall for any selected cycle
- Model performance panel
- Cost saving calculator